# 03 - Sistema hibrido: clasificador clasico + LLM

Caracteriza el punto de operacion de un sistema hibrido que resuelve con el
clasificador clasico los mensajes de **alta confianza** y **delega al LLM** solo
los de baja confianza. El objetivo es recuperar la exactitud del LLM pagando su
costo/latencia unicamente en una fraccion minima de mensajes.

El LLM se representa con las metricas empiricas reportadas en el primer informe
(`gpt-4o-mini` + RAG): exactitud 0.9705 y latencia media 3.23 s por mensaje.


In [1]:
import sys, os
sys.path.insert(0, '..')
import importlib, experimentos as ex
importlib.reload(ex)
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [2]:
df = ex.cargar()
X,y,lang = df['text'].values, df['intent'].values, df['lang'].values
strat=np.array([f'{a}|{b}' for a,b in zip(y,lang)]); idx=np.arange(len(X))
itr,ite=train_test_split(idx,test_size=0.15,random_state=ex.RANDOM_STATE,stratify=strat)
Xtr,Xte,ytr,yte=X[itr],X[ite],y[itr],y[ite]; lang_te=lang[ite]
# se elige el mejor clasificador calibrado (probabilidades para el umbral)
evals={n:ex.evaluar_modelo(n,m,Xtr,ytr,Xte,yte,lang_te) for n,m in ex.construir_modelos().items()}
mejor=max(evals.values(), key=lambda r:r['f1_macro'])
print('Clasificador base del hibrido:', mejor['nombre'], f"(F1={mejor['f1_macro']:.4f})")


Clasificador base del hibrido: Naive Bayes (F1=0.9649)


## 1. Curva costo-exactitud

Para cada umbral de confianza `tau`, los mensajes con confianza menor a `tau`
se delegan al LLM. Se estima la exactitud del sistema combinando los aciertos
del clasificador en los mensajes retenidos con la exactitud empirica del LLM en
los delegados.

In [3]:
LLM=ex.LLM
conf=mejor['proba'].max(axis=1); clases=np.array(mejor['clases'])
pred=clases[mejor['proba'].argmax(axis=1)]; correcto=(pred==yte); N=len(yte)
curva=[]
for tau in np.linspace(0,1,51):
    delega=conf<tau
    acc=(correcto[~delega].sum()+LLM['accuracy']*delega.sum())/N
    curva.append({'tau':tau,'frac_llm':delega.mean(),'acc':acc})
curva=pd.DataFrame(curva)
cand=curva[curva['acc']>=LLM['accuracy']]
op=cand.loc[cand['frac_llm'].idxmin()]
print(f"Punto de operacion: tau={op['tau']:.2f}  delega {op['frac_llm']:.1%} al LLM  "
      f"acc={op['acc']:.4f}  ahorro de llamadas={1-op['frac_llm']:.1%}")


Punto de operacion: tau=0.90  delega 3.0% al LLM  acc=0.9739  ahorro de llamadas=97.0%


In [4]:
fig,ax=plt.subplots(figsize=(7,4.2))
ax.plot(curva['frac_llm'],curva['acc'],'-o',ms=3,color='#3b6fb0',label='Hibrido')
ax.axhline(LLM['accuracy'],color='#27ae60',ls='--',label=f"LLM puro ({LLM['accuracy']:.3f})")
ax.axhline(mejor['acc'],color='#c0392b',ls=':',label=f"Clasico puro ({mejor['acc']:.3f})")
ax.scatter([op['frac_llm']],[op['acc']],s=120,color='k',zorder=5,label='Operacion')
ax.set_xlabel('Fraccion delegada al LLM'); ax.set_ylabel('Exactitud'); ax.legend()
plt.tight_layout(); plt.show()


## 2. Estimacion de ahorro de costo y latencia

Frente a un despliegue que envia **todos** los mensajes al LLM, el hibrido en su
punto de operacion procesa localmente la gran mayoria y solo paga el costo y la
latencia del LLM en la fraccion delegada.

In [5]:
frac=op['frac_llm']
lat_clasico_ms=mejor['lat_ms']
lat_hibrida=(1-frac)*lat_clasico_ms/1000 + frac*LLM['latency_mean_s']
lat_llm=LLM['latency_mean_s']
costo_rel=frac  # el clasico es de costo despreciable
print(f"Latencia media LLM puro : {lat_llm:.2f} s/mensaje")
print(f"Latencia media hibrido  : {lat_hibrida:.3f} s/mensaje  ({lat_llm/lat_hibrida:.0f}x mas rapido)")
print(f"Llamadas al LLM          : {frac:.1%} de los mensajes  (ahorro {1-frac:.1%})")


Latencia media LLM puro : 3.23 s/mensaje
Latencia media hibrido  : 0.098 s/mensaje  (33x mas rapido)
Llamadas al LLM          : 3.0% de los mensajes  (ahorro 97.0%)


## 3. Conclusion operativa

El sistema hibrido iguala o supera la exactitud del LLM puro delegando solo una
fraccion minima de los mensajes, con una latencia media casi dos ordenes de
magnitud menor. Esto confirma la hipotesis del proyecto: los modelos clasicos
cubren la etapa de deteccion de intencion con calidad equivalente y costo
despreciable, reservando el LLM para los casos genuinamente ambiguos.
